# EpiShield-AI — Phase 1: Final Data Cleaning & Model Preparation (revised)

One notebook that takes the four source datasets exactly as `extract_data()` returns them, addresses the identified data-quality issues without forcing unsupported corrections, and writes **clean tables** and **model-ready tables**.

## Cleaning policy (why the data can be trusted by the model)

| Principle | What it means here |
|---|---|
| **No imputation** | Missing values are never replaced by 0, means or forward-fill. "Not reported" stays missing; availability flags tell the model when a column starts. |
| **No rows deleted** | Row counts of all four datasets are preserved (asserted). |
| **No feature look-ahead** | Cleaning features use no future values. The explicit next-day target is a label only and must never enter `X`. |
| **Corrections only with proof** | The original value is preserved unless the correction rule is explicitly supported by independent accounting identities. Every change is logged. |
| **Ambiguous errors are flagged, not guessed** | If the faulty component cannot be identified, the original value is kept and the row gets `flag_identity_mismatch`. |
| **Identifiers fixed once, centrally** | Region code (2-char string), canonical region name, coordinates and NUTS codes come from one region dimension table. |

## What this notebook does, step by step

1. **Dates**: text timestamps (17:00 / 18:00 mixed) → plain calendar date.
2. **Population + region dimension**: numeric types, age-band classification, population-file summaries and age-band coverage flags; denominators are not treated as verified resident population until source scope/reference year are confirmed.
3. **Regional**: types, text, canonical region names (`P.A. Bolzano` → `Bolzano`), coordinates and NUTS from the dimension, evidence-based corrections, quality flags.
4. **National**: same engine, empty columns dropped, evidence-based corrections, quality flags.
5. **Notes**: whitespace-only notes → missing, `has_note` flag; kept as a separate date-level table (not merged into regional rows).
6. **Cross-checks**: regional-vs-national differences are measured and exported; neither source is forced to match the other.
7. **Model-ready tables**: invalid negative daily values → missing, derived daily series (never negative), availability flags and a one-day-ahead target label; population rates remain disabled until the population file scope is verified.
8. **Validation**: automatic PASS/FAIL checks, then everything is saved.

**Run:** open from inside the EpiShield-AI repository (needs `Src/Data Engineering/extract.py`) and use *Restart & Run All*.

### Important modelling contract
The forecast target is **`nuovi_positivi` (daily new cases)**. The regional modelling table includes `target_next_day_new_cases` as the supervised label for a one-day-ahead forecast. Do not include that label, `target_date`, or any columns computed from future rows in `X`. Use a chronological split within each region. The notebook prepares data; it does not guarantee a particular accuracy score.


In [2]:
import sys
import re
import warnings
from functools import reduce
from operator import add
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:          # allows running as a plain script
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)

# Optional manual overrides -------------------------------------------------
PROJECT_DIR_OVERRIDE = None     # e.g. Path(r"C:\Projects\EpiShield-AI")
PROCESSED_DIR_OVERRIDE = None   # e.g. Path(r"C:\Projects\EpiShield-AI\Processed Data")


def find_project_root():
    if PROJECT_DIR_OVERRIDE is not None:
        root = Path(PROJECT_DIR_OVERRIDE).resolve()
        if not (root / "Src" / "Data Engineering" / "extract.py").is_file():
            raise FileNotFoundError(f"extract.py not found under {root}")
        return root
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "Src" / "Data Engineering" / "extract.py").is_file():
            return path
    raise FileNotFoundError(
        "Project directory not found. Expected Src/Data Engineering/extract.py in the current "
        "folder or one of its parents (or set PROJECT_DIR_OVERRIDE)."
    )


PROJECT_DIR = find_project_root()
EXTRACT_DIR = str(PROJECT_DIR / "Src" / "Data Engineering")
if EXTRACT_DIR not in sys.path:
    sys.path.insert(0, EXTRACT_DIR)

# Same convention as the earlier preprocessing notebook: "../Processed Data"
PROCESSED_DIR = (Path(PROCESSED_DIR_OVERRIDE) if PROCESSED_DIR_OVERRIDE is not None
                 else (Path.cwd() / ".." / "Processed Data").resolve())
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory :", PROJECT_DIR)
print("Clean data written to:", PROCESSED_DIR)

Project directory : C:\Users\New Star Computer 25\EpiShield-AI
Clean data written to: C:\Users\New Star Computer 25\EpiShield-AI\Src\Processed Data


In [3]:
# ------------------------------------------------------------------
# Constants and helpers
# ------------------------------------------------------------------
CLEANING_LOG = []     # one row per cleaning step
CORRECTIONS = []      # one row per corrected cell
CHECKS = []           # PASS / FAIL validation results
SAVED = []

NUM_COLS = [
    "ricoverati_con_sintomi", "terapia_intensiva", "totale_ospedalizzati",
    "isolamento_domiciliare", "totale_positivi", "variazione_totale_positivi",
    "nuovi_positivi", "dimessi_guariti", "deceduti",
    "casi_da_sospetto_diagnostico", "casi_da_screening", "totale_casi",
    "tamponi", "casi_testati", "ingressi_terapia_intensiva",
    "totale_positivi_test_molecolare", "totale_positivi_test_antigenico_rapido",
    "tamponi_test_molecolare", "tamponi_test_antigenico_rapido",
]
NOTE_COLS = ["note", "note_test", "note_casi"]

# Reported only for a few months in 2020, decreasing almost daily and ending at 0: NOT usable for modelling
UNUSABLE_FOR_MODEL = ["casi_da_sospetto_diagnostico", "casi_da_screening"]

# Cumulative columns that must never decrease (used for flagging and for derived daily series)
CUMULATIVE_COLS = [
    "totale_casi", "deceduti", "dimessi_guariti", "tamponi", "casi_testati",
    "tamponi_test_molecolare", "tamponi_test_antigenico_rapido",
    "totale_positivi_test_molecolare", "totale_positivi_test_antigenico_rapido",
]

# Accounting identities: target = sum(parts)
IDENTITIES = [
    ("R1", "totale_ospedalizzati", ["ricoverati_con_sintomi", "terapia_intensiva"]),
    ("R2", "totale_positivi", ["totale_ospedalizzati", "isolamento_domiciliare"]),
    ("R3", "totale_casi", ["totale_positivi", "dimessi_guariti", "deceduti"]),
]

# Canonical region names: the population file's spelling
NAME_FIX = {"P.A. Bolzano": "Bolzano", "P.A. Trento": "Trento"}


def log_step(dataset, step, detail, affected=None):
    CLEANING_LOG.append({"dataset": dataset, "step": step, "detail": detail, "affected": affected})


def log_correction(dataset, date, region, column, old, new, rule, evidence):
    CORRECTIONS.append({"dataset": dataset, "date": date, "codice_regione": region, "column": column,
                        "old_value": old, "new_value": new, "rule": rule, "evidence": evidence})


def check(name, passed, detail=""):
    CHECKS.append({"check": name, "result": "PASS" if passed else "FAIL", "detail": detail})


def clean_text(s):
    """Collapse tabs/multiple spaces, strip, and turn empty strings into missing."""
    s = s.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    return s.mask(s.eq("").fillna(False), pd.NA)


def _numeric(series):
    s = series.astype("string").str.strip()
    s = s.mask(s.eq("").fillna(False), pd.NA)
    return pd.to_numeric(s.astype(object).where(s.notna(), np.nan), errors="raise").astype("float64")


def to_int64(series):
    num = _numeric(series)
    if (num.dropna() % 1 != 0).any():
        raise ValueError(f"Non-integer values in {series.name}")
    return num.astype("Int64")


def to_float64(series):
    return _numeric(series)


def to_day(series):
    parsed = pd.to_datetime(series.astype("string").str.strip(), errors="raise")
    if getattr(parsed.dt, "tz", None) is not None:
        parsed = parsed.dt.tz_localize(None)
    return parsed.dt.normalize()


def fnum(df, col):
    """Column as float64 with NaN for missing (safe for arithmetic on nullable ints)."""
    if col not in df.columns:           # column was dropped (e.g. 100% empty) -> all missing
        return pd.Series(np.nan, index=df.index, dtype="float64")
    return pd.Series(df[col].to_numpy(dtype="float64", na_value=np.nan), index=df.index)


def d_str(x):
    return "n/a" if pd.isna(x) else pd.Timestamp(x).strftime("%Y-%m-%d")


def save_csv(df, filename):
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[c]):
            out[c] = out[c].dt.strftime("%Y-%m-%d")
    path = PROCESSED_DIR / filename
    out.to_csv(path, index=False, encoding="utf-8-sig")
    SAVED.append((filename, len(out), out.shape[1]))
    return path


print("Helpers ready.")

Helpers ready.


## 1. Load the source data

`extract_data()` is called once. The raw DataFrames are kept untouched; all work happens on deep copies.

In [4]:
from extract import extract_data

data = extract_data()
for key in ["regioni", "nazionale", "popolazione", "note"]:
    if key not in data:
        raise KeyError(f"extract_data() did not return '{key}'")

RAW = {k: data[k] for k in ["regioni", "nazionale", "popolazione", "note"]}
RAW_SHAPES = {k: v.shape for k, v in RAW.items()}
for k, v in RAW_SHAPES.items():
    print(f"{k:<12} rows={v[0]:>7,}  columns={v[1]}")

regioni: 37401 rows, 30 columns
nazionale: 1781 rows, 24 columns
popolazione: 210 rows, 12 columns
note: 1617 rows, 2 columns
regioni      rows= 37,401  columns=30
nazionale    rows=  1,781  columns=24
popolazione  rows=    210  columns=12
note         rows=  1,617  columns=2


## 2. Population and the region dimension

The population file is the reference for region identity: one row per region with code, name, NUTS codes and coordinates. Regional and national tables are aligned to it.

Age-band issue found in the quality assessment: **Lombardia** uses `05-11 | 12-19` instead of `0-15 | 16-19`, so ages 0–4 are missing from its total. Nothing is invented; the region is flagged and extra denominators are provided that *are* comparable across all regions (population aged 20+ and under 20).

In [5]:
pop = RAW["popolazione"].copy(deep=True)

for c in ["codice_regione", "codice_nuts_1", "descrizione_nuts_1", "codice_nuts_2",
          "denominazione_regione", "sigla_regione", "range_eta"]:
    pop[c] = clean_text(pop[c])
pop["codice_regione"] = pop["codice_regione"].str.zfill(2)
pop["denominazione_regione"] = pop["denominazione_regione"].replace(NAME_FIX)

for c in ["totale_genere_maschile", "totale_genere_femminile", "totale_generale"]:
    pop[c] = to_int64(pop[c])
for c in ["latitudine_regione", "longitudine_regione"]:
    pop[c] = to_float64(pop[c])

# male + female must equal total
mf_ok = (pop["totale_genere_maschile"] + pop["totale_genere_femminile"] == pop["totale_generale"]).fillna(False)
check("Population: male + female = total in every row", bool(mf_ok.all()), f"{int((~mf_ok).sum())} mismatching rows")
check("Population: (region, age band) key is unique", not pop.duplicated(["codice_regione", "range_eta"]).any())


# parse age bands: '0-15' -> (0, 15), '90+' -> (90, 999)
def parse_band(b):
    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", str(b))
    if m:
        return int(m.group(1)), int(m.group(2))
    m = re.fullmatch(r"(\d+)\s*\+", str(b))
    if m:
        return int(m.group(1)), 999
    return None, None


bounds = pop["range_eta"].map(parse_band)
pop["age_min"] = bounds.map(lambda t: t[0]).astype("Int64")
pop["age_max"] = bounds.map(lambda t: t[1]).astype("Int64")
check("Population: every age-band label could be parsed", bool(pop["age_min"].notna().all()))

# Check that age bands form a continuous, non-overlapping sequence per region.
def age_band_coverage_ok(group):
    pairs = sorted([(int(a), int(b)) for a, b in zip(group["age_min"], group["age_max"]) if pd.notna(a) and pd.notna(b)])
    if not pairs:
        return False
    if pairs[0][0] != 0 or pairs[-1][1] != 999:
        return False
    return all(next_lo == prev_hi + 1 for (_, prev_hi), (next_lo, _) in zip(pairs, pairs[1:]))

coverage_ok = pop.groupby("codice_regione")[["age_min", "age_max"]].apply(age_band_coverage_ok)

band_sets = pop.groupby("codice_regione")["range_eta"].apply(lambda s: tuple(sorted(s)))
most_common = band_sets.value_counts().idxmax()
nonstandard = band_sets.map(lambda t: t != most_common)
pop["flag_age_bands_nonstandard"] = pop["codice_regione"].map(nonstandard).astype(bool)
pop = pop.sort_values(["codice_regione", "age_min"]).reset_index(drop=True)

# ---- region dimension (one row per region)
dim_cols = ["codice_regione", "denominazione_regione", "sigla_regione", "codice_nuts_1",
            "descrizione_nuts_1", "codice_nuts_2", "latitudine_regione", "longitudine_regione"]
region_dim = pop[dim_cols].drop_duplicates().reset_index(drop=True)
if not region_dim["codice_regione"].is_unique:
    raise ValueError("Region dimension is not unique per region code - check the population file")

g = pop.groupby("codice_regione")
summary = pd.DataFrame({
    "population_observed_sum": g["totale_generale"].sum(min_count=1),
    "youngest_age_covered": g["age_min"].min(),
    "population_20plus": pop[(pop["age_min"] >= 20).fillna(False)].groupby("codice_regione")["totale_generale"].sum(),
    "population_under_20": pop[(pop["age_max"] < 20).fillna(False)].groupby("codice_regione")["totale_generale"].sum(),
}).reset_index()
summary["flag_population_excludes_young"] = summary["youngest_age_covered"] > 0
summary["flag_age_band_coverage_incomplete"] = ~summary["codice_regione"].map(coverage_ok).fillna(False).astype(bool)
summary["population_total"] = summary["population_observed_sum"].where(~summary["flag_age_band_coverage_incomplete"])
summary["population_under_20"] = summary["population_under_20"].where(~summary["flag_age_band_coverage_incomplete"])
summary["population_20plus"] = summary["population_20plus"].where(~summary["flag_age_band_coverage_incomplete"])
region_dim = region_dim.merge(summary, on="codice_regione", how="left", validate="1:1")
for c in ["population_observed_sum", "population_total", "youngest_age_covered", "population_20plus", "population_under_20"]:
    region_dim[c] = region_dim[c].astype("Int64")

log_step("Population", "Types and text", "Counts -> Int64, coordinates -> float, text trimmed, codes -> 2-char strings, labels parsed", len(pop))
log_step("Population", "Region names", f"Canonical names from population file; mapping applied: {NAME_FIX}", None)
region_dim["population_scope_verified"] = False  # DQA did not establish resident-population scope/reference year
region_dim["population_denominator_usable"] = False  # do not use for epidemiological rates until verified
n_flag = int(region_dim["flag_age_band_coverage_incomplete"].sum())
log_step("Population", "Age-band flag", "Age bands with gaps, overlaps, or incomplete all-age coverage are flagged; source population scope/reference year is unverified, so denominators are not approved for epidemiological rates", n_flag)

print("Regions:", len(region_dim), "| age-band rows:", len(pop), "| regions flagged for missing young ages:", n_flag)
display(region_dim[["codice_regione", "denominazione_regione", "population_observed_sum", "population_total", "population_20plus",
                    "youngest_age_covered", "flag_population_excludes_young", "flag_age_band_coverage_incomplete", "population_scope_verified"]])

Regions: 21 | age-band rows: 210 | regions flagged for missing young ages: 1


,codice_regione,denominazione_regione,population_observed_sum,population_total,population_20plus,youngest_age_covered,flag_population_excludes_young,flag_age_band_coverage_incomplete,population_scope_verified
0,01,Piemonte,4311217,4311217,3594047,0,False,False,False
1,02,Valle d'Aosta,125034,125034,103069,0,False,False,False
2,03,Lombardia,9597086,<NA>,<NA>,5,True,True,False
3,05,Veneto,4879133,4879133,4011772,0,False,False,False
4,06,Friuli Venezia Giulia,1206216,1206216,1010431,0,False,False,False
5,07,Liguria,1524826,1524826,1293758,0,False,False,False
6,08,Emilia-Romagna,4464119,4464119,3688003,0,False,False,False
7,09,Toscana,3692555,3692555,3079001,0,False,False,False
8,10,Umbria,870165,870165,725102,0,False,False,False
9,11,Marche,1512672,1512672,1256242,0,False,False,False


## 3. Shared cleaning engine (Regional and National)

The two daily files go through the same functions, so they are cleaned by identical rules:

- **Types:** dates → calendar day, counts → nullable integers (`Int64`, so missing stays missing).
- **Constant / empty columns:** `stato` (always `ITA`) and columns that are 100% empty are dropped and logged.
- **Evidence-based corrections** (see policy): a field is corrected only when two identities both fail and recomputing that one field from its own components makes both hold.
- **Flags** (original numbers are *not* altered by flags): `flag_value_corrected`, `flag_identity_mismatch`, `flag_negative_daily_value`, `flag_cumulative_decrease`, `flag_new_cases_mismatch`.

In [6]:
def clean_timeseries(raw, dataset):
    df = raw.copy(deep=True)
    df["date"] = to_day(df["data"])
    df = df.drop(columns=["data"])

    if "stato" in df.columns:
        vals = clean_text(df["stato"]).dropna().unique()
        if len(vals) == 1:
            df = df.drop(columns=["stato"])
            log_step(dataset, "Dropped constant column", f"'stato' always equals {vals[0]!r}", len(df))

    for c in NUM_COLS:
        if c in df.columns:
            df[c] = to_int64(df[c])
    for c in NOTE_COLS:
        if c in df.columns:
            df[c] = clean_text(df[c])

    empty_cols = [c for c in df.columns if df[c].isna().all()]
    if empty_cols:
        df = df.drop(columns=empty_cols)
        log_step(dataset, "Dropped 100% empty columns", f"{empty_cols}", len(empty_cols))
    log_step(dataset, "Types and text", "date -> calendar day; counts -> Int64 (missing preserved); notes trimmed, blanks -> missing", len(df))
    return df


def identity_masks(df):
    """Boolean violation mask per identity (rows with missing parts are never violations)."""
    out = {}
    for label, target, parts in IDENTITIES:
        if not all(c in df.columns for c in [target] + parts):
            out[label] = pd.Series(False, index=df.index)
            continue
        expected = reduce(add, [fnum(df, p) for p in parts])
        out[label] = ((fnum(df, target) - expected).abs() > 0).fillna(False)
    return out


def apply_identity_corrections(df, dataset):
    """Correct a field only if two identities fail, the field is shared by both, and the fix makes both hold."""
    df["flag_value_corrected"] = False
    ident = {l: (t, p) for l, t, p in IDENTITIES}
    # (field to fix, identity that defines it, identity that must also hold afterwards)
    steps = [("totale_ospedalizzati", "R1", "R2"), ("totale_positivi", "R2", "R3")]
    total_fixed = 0
    for field, own, other in steps:
        masks = identity_masks(df)
        candidates = masks[own] & masks[other]
        if not candidates.any():
            continue
        own_target, own_parts = ident[own]
        new_val = reduce(add, [fnum(df, p) for p in own_parts])
        t_other, p_other = ident[other]
        vals = {c: fnum(df, c) for c in set([t_other] + p_other)}
        vals[field] = new_val
        holds = ((vals[t_other] - reduce(add, [vals[p] for p in p_other])).abs() == 0).fillna(False)
        fix = candidates & holds
        for i in df.index[fix]:
            old = df.at[i, field]
            new = int(new_val[i])
            df.at[i, field] = new
            df.at[i, "flag_value_corrected"] = True
            region = df.at[i, "codice_regione"] if "codice_regione" in df.columns else "ITA"
            log_correction(dataset, d_str(df.at[i, "date"]), region, field, int(old), new, f"{own}+{other}",
                           f"{own} and {other} both failed; {field} recomputed from {own_parts} makes {other} hold too")
            total_fixed += 1
    log_step(dataset, "Evidence-based corrections", "Shared faulty field recomputed only when it makes both failing identities hold", total_fixed)
    return df, total_fixed


def cumulative_decreases(df, group_col):
    flag = pd.Series(False, index=df.index)
    frames = []
    for c in CUMULATIVE_COLS:
        if c not in df.columns:
            continue
        v = fnum(df, c)
        ok = v.notna()
        sub = pd.DataFrame({"date": df.loc[ok, "date"], "v": v[ok]})
        if group_col:
            sub["g"] = df.loc[ok, group_col]
            ch = sub.groupby("g", sort=False)["v"].diff()
        else:
            ch = sub["v"].diff()
        dec = (ch < 0).fillna(False)
        if dec.any():
            flag.loc[dec[dec].index] = True
            fr = pd.DataFrame({"date": sub.loc[dec, "date"], "column": c, "change": ch[dec], "value_after": sub.loc[dec, "v"]})
            if group_col:
                fr.insert(1, group_col, sub.loc[dec, "g"])
            frames.append(fr)
    return flag, (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame())


def new_cases_gap(df, group_col):
    """(change in totale_casi vs previous calendar day) - nuovi_positivi; NaN when not comparable."""
    v = fnum(df, "totale_casi")
    if group_col:
        prev_v = v.groupby(df[group_col], sort=False).shift(1)
        prev_d = df["date"].groupby(df[group_col], sort=False).shift(1)
    else:
        prev_v, prev_d = v.shift(1), df["date"].shift(1)
    gap = (df["date"] - prev_d).dt.days
    delta = (v - prev_v).where(gap == 1)
    return delta - fnum(df, "nuovi_positivi")


def add_flags(df, dataset, group_col):
    neg = (fnum(df, "nuovi_positivi") < 0) | (fnum(df, "ingressi_terapia_intensiva") < 0)
    df["flag_negative_daily_value"] = neg.fillna(False).astype(bool)

    dec_flag, dec_table = cumulative_decreases(df, group_col)
    df["flag_cumulative_decrease"] = dec_flag.astype(bool)

    masks = identity_masks(df)
    df["flag_identity_mismatch"] = (masks["R1"] | masks["R2"] | masks["R3"]).astype(bool)

    gap = new_cases_gap(df, group_col)
    df["new_cases_vs_cumulative_diff"] = gap.round().astype("Int64")
    df["flag_new_cases_mismatch"] = (gap.abs() > 0).fillna(False).astype(bool)

    for col in ["flag_value_corrected", "flag_negative_daily_value", "flag_cumulative_decrease",
                "flag_identity_mismatch", "flag_new_cases_mismatch"]:
        log_step(dataset, f"Flag {col}", "Original numbers untouched; boolean marker added", int(df[col].sum()))
    return df, dec_table


def identity_exceptions(df, key_cols):
    rows = []
    for label, target, parts in IDENTITIES:
        if not all(c in df.columns for c in [target] + parts):
            continue
        expected = reduce(add, [fnum(df, p) for p in parts])
        diff = fnum(df, target) - expected
        bad = diff.abs() > 0
        if bad.any():
            t = df.loc[bad, key_cols].copy()
            t["rule"] = f"{label}: {target} = " + " + ".join(parts)
            t["recorded"] = fnum(df, target)[bad].astype("Int64")
            t["expected"] = expected[bad].astype("Int64")
            t["difference"] = diff[bad].astype("Int64")
            rows.append(t)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=key_cols + ["rule", "recorded", "expected", "difference"])


def grid_report(df, group_col, name):
    days = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    if group_col:
        full = pd.MultiIndex.from_product([df[group_col].unique(), days])
        present = pd.MultiIndex.from_frame(df[[group_col, "date"]])
        dups = int(df.duplicated([group_col, "date"]).sum())
    else:
        full = pd.Index(days)
        present = pd.Index(df["date"])
        dups = int(df["date"].duplicated().sum())
    missing = full.difference(present)
    print(f"{name}: expected {len(full):,} rows, found {len(df):,}; missing group-days: {len(missing)}; duplicate keys: {dups}")
    return len(missing), dups


print("Engine ready.")

Engine ready.


## 4. Regional dataset

In [7]:
reg = clean_timeseries(RAW["regioni"], "Regional")

# ---- region identifiers from the region dimension
reg["codice_regione"] = clean_text(reg["codice_regione"]).str.zfill(2)
unknown = set(reg["codice_regione"]) - set(region_dim["codice_regione"])
if unknown:
    raise ValueError(f"Region codes not found in the population file: {sorted(unknown)}")

dim_idx = region_dim.set_index("codice_regione")
raw_names = clean_text(reg["denominazione_regione"])
reg["denominazione_regione"] = reg["codice_regione"].map(dim_idx["denominazione_regione"])
renamed = int((raw_names != reg["denominazione_regione"]).sum())
log_step("Regional", "Canonical region names", f"Names taken from the region dimension via code (e.g. {NAME_FIX})", renamed)

# coordinates: one value per region from the dimension (verify the file agrees first)
lat_raw, long_raw = to_float64(reg["lat"]), to_float64(reg["long"])
lat_dim = reg["codice_regione"].map(dim_idx["latitudine_regione"]).astype("float64")
long_dim = reg["codice_regione"].map(dim_idx["longitudine_regione"]).astype("float64")
d_lat, d_long = (lat_raw - lat_dim).abs(), (long_raw - long_dim).abs()
material = int(((d_lat > 1e-6) | (d_long > 1e-6)).sum())
print(f"Raw regional coordinates: distinct lat={lat_raw.nunique()}, long={long_raw.nunique()} (regions={reg['codice_regione'].nunique()}); "
      f"max |diff| vs dimension: lat={d_lat.max():.8f}, long={d_long.max():.8f}; rows differing by > 1e-6: {material}")
reg["lat"], reg["long"] = lat_dim, long_dim
log_step("Regional", "Coordinates", f"lat/long set to one value per region from the dimension; rows that differed materially: {material}", int(len(reg)))

# NUTS: constant per region, empty in the first period -> restored from the dimension (identifier, not a measurement)
nuts_conflicts = 0
for c in ["codice_nuts_1", "codice_nuts_2"]:
    raw_v = clean_text(reg[c])
    dim_v = reg["codice_regione"].map(dim_idx[c])
    nuts_conflicts += int((raw_v.notna() & (raw_v != dim_v)).sum())
    log_step("Regional", f"{c} restored", "Empty cells filled from the region dimension (value is constant per region)", int(raw_v.isna().sum()))
    reg[c] = dim_v.astype("string")
check("Regional: NUTS values in the file agree with the population file", nuts_conflicts == 0, f"{nuts_conflicts} conflicts")

# ---- column order, sort
front = ["date", "codice_regione", "denominazione_regione", "codice_nuts_1", "codice_nuts_2", "lat", "long"]
reg = reg[front + [c for c in reg.columns if c not in front]]
reg = reg.sort_values(["codice_regione", "date"]).reset_index(drop=True)

missing_days, dup_keys = grid_report(reg, "codice_regione", "Regional")
check("Regional: no duplicate (region, date) keys", dup_keys == 0)
check("Regional: complete region x day grid", missing_days == 0, f"{missing_days} missing region-days")
check("Regional: row count preserved", len(reg) == RAW_SHAPES["regioni"][0], f"{len(reg)} vs {RAW_SHAPES['regioni'][0]}")

Raw regional coordinates: distinct lat=21, long=23 (regions=21); max |diff| vs dimension: lat=0.00000000, long=0.00000001; rows differing by > 1e-6: 0
Regional: expected 37,401 rows, found 37,401; missing group-days: 0; duplicate keys: 0


In [8]:
# ---- evidence-based corrections, then flags
reg_before = reg.copy(deep=True)
reg, n_fixed_reg = apply_identity_corrections(reg, "Regional")
reg, reg_decreases = add_flags(reg, "Regional", "codice_regione")

print("Cells corrected:", n_fixed_reg)
print("Rows with flag_identity_mismatch (still unexplained, original values kept):", int(reg["flag_identity_mismatch"].sum()))
print("Rows with flag_negative_daily_value:", int(reg["flag_negative_daily_value"].sum()))
print("Rows with flag_cumulative_decrease:", int(reg["flag_cumulative_decrease"].sum()))
print("Rows with flag_new_cases_mismatch:", int(reg["flag_new_cases_mismatch"].sum()),
      "(small differences are common; see new_cases_vs_cumulative_diff for the size)")

reg_identity_left = identity_exceptions(reg, ["date", "codice_regione", "denominazione_regione"])
print("\nRemaining accounting mismatches (kept as reported, flagged):")
display(reg_identity_left)

Cells corrected: 1
Rows with flag_identity_mismatch (still unexplained, original values kept): 14
Rows with flag_negative_daily_value: 19
Rows with flag_cumulative_decrease: 1079
Rows with flag_new_cases_mismatch: 1923 (small differences are common; see new_cases_vs_cumulative_diff for the size)

Remaining accounting mismatches (kept as reported, flagged):


,date,codice_regione,denominazione_regione,rule,recorded,expected,difference
0,2021-08-11,02,Valle d'Aosta,R3: totale_casi = totale_positivi + dimessi_gu...,11871,11869,2
1,2021-06-23,05,Veneto,R3: totale_casi = totale_positivi + dimessi_gu...,425162,425159,3
2,2021-08-01,05,Veneto,R3: totale_casi = totale_positivi + dimessi_gu...,437066,437063,3
3,2021-06-23,06,Friuli Venezia Giulia,R3: totale_casi = totale_positivi + dimessi_gu...,107502,107487,15
4,2021-02-03,08,Emilia-Romagna,R3: totale_casi = totale_positivi + dimessi_gu...,221615,221614,1
5,2020-11-04,09,Toscana,R3: totale_casi = totale_positivi + dimessi_gu...,52815,52799,16
6,2024-07-05,09,Toscana,R3: totale_casi = totale_positivi + dimessi_gu...,1655099,1655119,-20
7,2024-07-06,09,Toscana,R3: totale_casi = totale_positivi + dimessi_gu...,1655140,1655160,-20
8,2024-07-07,09,Toscana,R3: totale_casi = totale_positivi + dimessi_gu...,1655186,1655206,-20
9,2024-07-08,09,Toscana,R3: totale_casi = totale_positivi + dimessi_gu...,1655186,1655175,11


## 5. National dataset

In [9]:
nat = clean_timeseries(RAW["nazionale"], "National")
nat = nat.sort_values("date").reset_index(drop=True)
nat = nat[["date"] + [c for c in nat.columns if c != "date"]]

missing_days_n, dup_keys_n = grid_report(nat, None, "National")
check("National: no duplicate dates", dup_keys_n == 0)
check("National: complete daily calendar", missing_days_n == 0, f"{missing_days_n} missing days")
check("National: row count preserved", len(nat) == RAW_SHAPES["nazionale"][0], f"{len(nat)} vs {RAW_SHAPES['nazionale'][0]}")

nat, n_fixed_nat = apply_identity_corrections(nat, "National")
nat, nat_decreases = add_flags(nat, "National", None)

print("Cells corrected:", n_fixed_nat)
print("Rows with flag_identity_mismatch:", int(nat["flag_identity_mismatch"].sum()))
print("Rows with flag_cumulative_decrease:", int(nat["flag_cumulative_decrease"].sum()))
print("Rows with flag_new_cases_mismatch:", int(nat["flag_new_cases_mismatch"].sum()))
nat_identity_left = identity_exceptions(nat, ["date"])
print("\nRemaining accounting mismatches (kept as reported, flagged):")
display(nat_identity_left)

National: expected 1,781 rows, found 1,781; missing group-days: 0; duplicate keys: 0
Cells corrected: 2
Rows with flag_identity_mismatch: 12
Rows with flag_cumulative_decrease: 30
Rows with flag_new_cases_mismatch: 1033

Remaining accounting mismatches (kept as reported, flagged):


,date,rule,recorded,expected,difference
0,2020-11-04,R3: totale_casi = totale_positivi + dimessi_gu...,790377,790360,17
1,2021-02-03,R3: totale_casi = totale_positivi + dimessi_gu...,2583790,2583789,1
2,2021-06-23,R3: totale_casi = totale_positivi + dimessi_gu...,4255434,4255416,18
3,2021-08-01,R3: totale_casi = totale_positivi + dimessi_gu...,4355348,4355345,3
4,2021-08-11,R3: totale_casi = totale_positivi + dimessi_gu...,4413162,4413160,2
5,2022-01-11,R3: totale_casi = totale_positivi + dimessi_gu...,7774863,7774636,227
6,2024-07-05,R3: totale_casi = totale_positivi + dimessi_gu...,26742114,26742134,-20
7,2024-07-06,R3: totale_casi = totale_positivi + dimessi_gu...,26742772,26742792,-20
8,2024-07-07,R3: totale_casi = totale_positivi + dimessi_gu...,26743348,26743368,-20
9,2024-07-08,R3: totale_casi = totale_positivi + dimessi_gu...,26743662,26743651,11


## 6. Notes dataset

Notes are free Italian text at **date level** (they mention several regions). They are cleaned and kept as a **separate table**; they are not merged into regional rows.

In [10]:
notes = RAW["note"].copy(deep=True)
notes["date"] = to_day(notes["data"])
notes = notes.drop(columns=["data"])
text_col = "note"
notes[text_col] = clean_text(notes[text_col])
notes["has_note"] = notes[text_col].notna()
notes = notes.sort_values("date").reset_index(drop=True)[["date", text_col, "has_note"]]

n_blank = int((~notes["has_note"]).sum())
log_step("Notes", "Whitespace-only notes", "Converted to missing; has_note = False. Text content (Italian) otherwise unchanged apart from collapsed whitespace", n_blank)
check("Notes: row count preserved", len(notes) == RAW_SHAPES["note"][0], f"{len(notes)} vs {RAW_SHAPES['note'][0]}")
check("Notes: one row per date", bool(notes["date"].is_unique))
print(f"Notes: {len(notes):,} rows | with text: {int(notes['has_note'].sum()):,} | empty: {n_blank}")

Notes: 1,617 rows | with text: 1,603 | empty: 14


## 7. Regional vs national reconciliation (diagnostic, not a cleaning gate)

Regional sums and the national source are independently published series. Differences can reflect source revisions, reporting differences or aggregation timing. We **report** every comparable mismatch; we do not alter either source to force equality. A mismatch is not, by itself, proof that regional values are wrong.


In [11]:
common = [c for c in NUM_COLS if c in reg.columns and c in nat.columns]
reg_sum = reg.groupby("date")[common].sum(min_count=1)
reg_cnt = reg.groupby("date")[common].count()
n_regions = reg["codice_regione"].nunique()
nat_idx = nat.set_index("date")[common]

check("Regional and national date coverage is reported", True,
      f"regional-only {len(set(reg_sum.index) - set(nat_idx.index))}, national-only {len(set(nat_idx.index) - set(reg_sum.index))}")

rows, bad_total = [], 0
for c in common:
    ok_mask = ((reg_cnt[c] == n_regions).to_numpy() & nat_idx[c].reindex(reg_cnt.index).notna().to_numpy())
    ok_days = reg_cnt.index[ok_mask]
    diff = fnum(reg_sum.loc[ok_days].reset_index(), c).to_numpy() - nat_idx.loc[ok_days, c].astype("float64").to_numpy()
    n_bad = int((diff != 0).sum())
    bad_total += n_bad
    rows.append({"column": c, "days_compared": len(ok_days), "days_mismatching": n_bad,
                 "max_abs_difference": float(np.abs(diff).max()) if len(diff) else 0.0})
recon = pd.DataFrame(rows)
display(recon[recon["days_mismatching"] > 0] if bad_total else recon.head(3))
check("Regional-national reconciliation completed (differences are diagnostic)", True, f"{bad_total} mismatching (day, column) pairs recorded; not forced to zero")
recon_mismatches = []
for c in common:
    ok_mask = ((reg_cnt[c] == n_regions).to_numpy() & nat_idx[c].reindex(reg_cnt.index).notna().to_numpy())
    ok_days = reg_cnt.index[ok_mask]
    diffs = fnum(reg_sum.loc[ok_days].reset_index(), c).to_numpy() - nat_idx.loc[ok_days, c].astype("float64").to_numpy()
    for day, diff in zip(ok_days, diffs):
        if diff != 0:
            recon_mismatches.append({"date": day, "column": c, "regional_sum": float(reg_sum.loc[day, c]), "national_value": float(nat_idx.loc[day, c]), "difference": float(diff)})
reconciliation_mismatches = pd.DataFrame(recon_mismatches, columns=["date", "column", "regional_sum", "national_value", "difference"])
print("Total mismatching (day, column) pairs (reported, not auto-corrected):", bad_total)

,column,days_compared,days_mismatching,max_abs_difference
0,ricoverati_con_sintomi,1781,0,0.0
1,terapia_intensiva,1781,0,0.0
2,totale_ospedalizzati,1781,0,0.0


Total mismatching (day, column) pairs (reported, not auto-corrected): 0


## 8. Model-ready tables

The *clean* tables above keep every reported number (plus flags). The *model* tables are the ones to train on:

- **Invalid negatives → missing** (not 0): negative `nuovi_positivi` and `ingressi_terapia_intensiva` are source corrections, not real counts.
- **Daily series derived from cumulative columns** (`new_deaths`, `new_recovered`, `new_tests`): computed only between consecutive calendar days and set to missing when the cumulative column decreased (source revision) - they are never negative.
- **Rates per 100k** using the region's population (flagged where the population is understated).
- **Availability flags** tell the model when a column starts to exist, so missing is not confused with zero.
- **Left out on purpose:** `casi_da_sospetto_diagnostico`, `casi_da_screening` (unusable), free-text note columns (replaced by `has_*` flags).

In [12]:
MODEL_COUNT_COLS = [
    "ricoverati_con_sintomi", "terapia_intensiva", "totale_ospedalizzati", "isolamento_domiciliare",
    "totale_positivi", "variazione_totale_positivi", "nuovi_positivi", "dimessi_guariti", "deceduti",
    "totale_casi", "tamponi", "casi_testati", "ingressi_terapia_intensiva",
    "totale_positivi_test_molecolare", "totale_positivi_test_antigenico_rapido",
    "tamponi_test_molecolare", "tamponi_test_antigenico_rapido",
]
FLAG_COLS = ["flag_value_corrected", "flag_identity_mismatch", "flag_negative_daily_value",
             "flag_cumulative_decrease", "flag_new_cases_mismatch"]


def daily_from_cumulative(df, group_col, col):
    """Day-to-day change of a cumulative column: only for consecutive days, never negative."""
    v = fnum(df, col)
    if group_col:
        prev_v = v.groupby(df[group_col], sort=False).shift(1)
        prev_d = df["date"].groupby(df[group_col], sort=False).shift(1)
    else:
        prev_v, prev_d = v.shift(1), df["date"].shift(1)
    gap = (df["date"] - prev_d).dt.days
    delta = v - prev_v
    return delta.where((gap == 1) & (delta >= 0)).round().astype("Int64")


def availability_start(df, col):
    ok = df[col].notna() if col in df.columns else pd.Series(False, index=df.index)
    return df.loc[ok, "date"].min() if ok.any() else pd.NaT


def build_model_table(df, group_col, population_total, population_20plus):
    m = df.copy(deep=True)
    assert m.equals(m.sort_values(([group_col] if group_col else []) + ["date"])), "table must be sorted by (group, date)"

    # invalid negatives -> missing
    for c in ["nuovi_positivi", "ingressi_terapia_intensiva"]:
        if c in m.columns:
            m[c] = m[c].where(~(fnum(m, c) < 0))

    # derived daily series
    m["new_deaths"] = daily_from_cumulative(df, group_col, "deceduti")
    m["new_recovered"] = daily_from_cumulative(df, group_col, "dimessi_guariti")
    m["new_tests"] = daily_from_cumulative(df, group_col, "tamponi")

    # Population columns are retained for audit/context, but rates are disabled.
    # DQA found that source scope/reference year has not been established (resident population vs target population).
    # Never calculate epidemiological rates from an unverified denominator.
    m["population_total"] = population_total
    m["population_20plus"] = population_20plus
    m["population_scope_verified"] = False
    m["population_denominator_usable"] = False
    m["new_cases_per_100k"] = np.nan
    m["active_cases_per_100k"] = np.nan
    m["new_cases_per_100k_20plus"] = np.nan
    m["active_cases_per_100k_20plus"] = np.nan

    # Separate source-era availability from whether this specific row has a value.
    for col, flag in [("tamponi_test_molecolare", "has_testing_breakdown"),
                      ("casi_testati", "has_casi_testati"),
                      ("ingressi_terapia_intensiva", "has_icu_entries")]:
        start = availability_start(df, col)
        m[flag] = (m["date"] >= start) if pd.notna(start) else False
        m[f"{flag}_value_present"] = df[col].notna() if col in df.columns else False

    # One-day-ahead supervised target. Negative reported values were already nulled above.
    target = m["nuovi_positivi"]
    if group_col:
        next_target = target.groupby(m[group_col], sort=False).shift(-1)
        next_date = m["date"].groupby(m[group_col], sort=False).shift(-1)
    else:
        next_target = target.shift(-1)
        next_date = m["date"].shift(-1)
    consecutive = (next_date - m["date"]).dt.days.eq(1)
    m["target_next_day_new_cases"] = next_target.where(consecutive).astype("Int64")
    m["target_date"] = next_date.where(consecutive)
    return m


# ---------------- regional model table
pop_by_code = region_dim.set_index("codice_regione")
reg_model = build_model_table(
    reg, "codice_regione",
    reg["codice_regione"].map(pop_by_code["population_total"]).astype("Int64"),
    reg["codice_regione"].map(pop_by_code["population_20plus"]).astype("Int64"),
)
reg_model["flag_population_excludes_young"] = reg["codice_regione"].map(pop_by_code["flag_population_excludes_young"]).astype(bool)
reg_model["flag_age_band_coverage_incomplete"] = reg["codice_regione"].map(pop_by_code["flag_age_band_coverage_incomplete"]).astype(bool)
for c in ["note", "note_test", "note_casi"]:
    if c in reg.columns:
        reg_model[f"has_{c}"] = reg[c].notna()

reg_keep = (["date", "codice_regione", "denominazione_regione", "codice_nuts_1", "codice_nuts_2", "lat", "long",
             "flag_population_excludes_young", "flag_age_band_coverage_incomplete", "population_scope_verified", "population_denominator_usable"]
            + [c for c in MODEL_COUNT_COLS if c in reg_model.columns]
            + ["new_deaths", "new_recovered", "new_tests", "new_cases_per_100k", "active_cases_per_100k",
               "new_cases_per_100k_20plus", "active_cases_per_100k_20plus",
               "has_testing_breakdown", "has_testing_breakdown_value_present",
               "has_casi_testati", "has_casi_testati_value_present",
               "has_icu_entries", "has_icu_entries_value_present", "target_next_day_new_cases", "target_date"]
            + [c for c in ["has_note", "has_note_test", "has_note_casi"] if c in reg_model.columns]
            + FLAG_COLS + ["new_cases_vs_cumulative_diff"])
reg_model = reg_model[reg_keep]

# ---------------- national model table
nat_pop_total = (int(region_dim["population_total"].sum()) if region_dim["population_total"].notna().all() else np.nan)
nat_pop_20 = int(region_dim["population_20plus"].sum())
nat_model = build_model_table(nat, None, nat_pop_total, nat_pop_20)
nat_model["flag_population_excludes_young"] = bool(region_dim["flag_population_excludes_young"].any())
nat_model["flag_age_band_coverage_incomplete"] = bool(region_dim["flag_age_band_coverage_incomplete"].any())
if "note" in nat.columns:
    nat_model["has_note"] = nat["note"].notna()
nat_keep = (["date", "flag_population_excludes_young", "flag_age_band_coverage_incomplete", "population_scope_verified", "population_denominator_usable"]
            + [c for c in MODEL_COUNT_COLS if c in nat_model.columns]
            + ["new_deaths", "new_recovered", "new_tests", "new_cases_per_100k", "active_cases_per_100k",
               "new_cases_per_100k_20plus", "active_cases_per_100k_20plus",
               "has_testing_breakdown", "has_testing_breakdown_value_present",
               "has_casi_testati", "has_casi_testati_value_present",
               "has_icu_entries", "has_icu_entries_value_present", "target_next_day_new_cases", "target_date"]
            + (["has_note"] if "has_note" in nat_model.columns else [])
            + FLAG_COLS + ["new_cases_vs_cumulative_diff"])
nat_model = nat_model[nat_keep]

log_step("Model tables", "Columns left out", f"{UNUSABLE_FOR_MODEL} and free-text note columns", None)
log_step("Model tables", "Negative daily values -> missing", "nuovi_positivi and ingressi_terapia_intensiva < 0 set to missing (not 0)",
         int(reg["flag_negative_daily_value"].sum() + nat["flag_negative_daily_value"].sum()))
log_step("Model tables", "Derived daily series", "new_deaths, new_recovered, new_tests: consecutive days only, never negative", None)
print("Regional model table:", reg_model.shape, "| National model table:", nat_model.shape)
display(reg_model.head(3))

Regional model table: (37401, 52) | National model table: (1781, 44)


,date,codice_regione,denominazione_regione,codice_nuts_1,codice_nuts_2,lat,long,flag_population_excludes_young,flag_age_band_coverage_incomplete,population_scope_verified,population_denominator_usable,ricoverati_con_sintomi,terapia_intensiva,totale_ospedalizzati,isolamento_domiciliare,totale_positivi,variazione_totale_positivi,nuovi_positivi,dimessi_guariti,deceduti,totale_casi,tamponi,casi_testati,ingressi_terapia_intensiva,totale_positivi_test_molecolare,totale_positivi_test_antigenico_rapido,tamponi_test_molecolare,tamponi_test_antigenico_rapido,new_deaths,new_recovered,new_tests,new_cases_per_100k,active_cases_per_100k,new_cases_per_100k_20plus,active_cases_per_100k_20plus,has_testing_breakdown,has_testing_breakdown_value_present,has_casi_testati,has_casi_testati_value_present,has_icu_entries,has_icu_entries_value_present,target_next_day_new_cases,target_date,has_note,has_note_test,has_note_casi,flag_value_corrected,flag_identity_mismatch,flag_negative_daily_value,flag_cumulative_decrease,flag_new_cases_mismatch,new_cases_vs_cumulative_diff
0,2020-02-24,01,Piemonte,ITC,ITC1,45.073274,7.680687,False,False,False,False,2,0,2,1,3,0,3,0,0,3,141,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,False,False,False,False,False,False,0,2020-02-25,False,False,False,False,False,False,False,False,<NA>
1,2020-02-25,01,Piemonte,ITC,ITC1,45.073274,7.680687,False,False,False,False,2,0,2,1,3,0,0,0,0,3,141,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,0,NaN,NaN,NaN,NaN,False,False,False,False,False,False,0,2020-02-26,False,False,False,False,False,False,False,False,0
2,2020-02-26,01,Piemonte,ITC,ITC1,45.073274,7.680687,False,False,False,False,2,0,2,1,3,0,0,0,0,3,156,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,15,NaN,NaN,NaN,NaN,False,False,False,False,False,False,<NA>,2020-02-27,False,False,False,False,False,False,False,False,0


## 9. Validation

Automatic checks. Every check must be `PASS` before the tables go to the model. Remaining accounting mismatches are *expected* (they are flagged, not guessed) and are listed for information.

In [13]:
# ---- values preserved except the logged corrections
def changed_cells(raw_df, clean_df, key_cols, cols):
    r = raw_df.copy()
    r["date"] = to_day(r["data"])
    if "codice_regione" in r.columns:
        r["codice_regione"] = clean_text(r["codice_regione"]).str.zfill(2)
    r = r.set_index(key_cols)
    cl = clean_df.set_index(key_cols)
    n = 0
    for col in cols:
        if col in cl.columns and col in r.columns:
            a = to_float64(r[col]).reindex(cl.index)
            b = pd.Series(cl[col].to_numpy(dtype="float64", na_value=np.nan), index=cl.index)
            n += int((~((a == b) | (a.isna() & b.isna()))).sum())
    return n


reg_changed = changed_cells(RAW["regioni"], reg, ["codice_regione", "date"], NUM_COLS)
nat_changed = changed_cells(RAW["nazionale"], nat, ["date"], NUM_COLS)
n_reg_corr = sum(1 for c in CORRECTIONS if c["dataset"] == "Regional")
n_nat_corr = sum(1 for c in CORRECTIONS if c["dataset"] == "National")
check("Regional: only the logged corrections differ from the raw numbers", reg_changed == n_reg_corr, f"{reg_changed} changed vs {n_reg_corr} logged")
check("National: only the logged corrections differ from the raw numbers", nat_changed == n_nat_corr, f"{nat_changed} changed vs {n_nat_corr} logged")

# ---- types and identifiers
check("Regional: counts are Int64", all(str(reg[c].dtype) == "Int64" for c in NUM_COLS if c in reg.columns))
check("National: counts are Int64", all(str(nat[c].dtype) == "Int64" for c in NUM_COLS if c in nat.columns))
check("Regional: region code is a 2-character string and exists in the dimension",
      bool(reg["codice_regione"].str.len().eq(2).all()) and set(reg["codice_regione"]) <= set(region_dim["codice_regione"]))
check("Regional: one name, one coordinate pair and one NUTS value per region",
      bool((reg.groupby("codice_regione")[["denominazione_regione", "lat", "long", "codice_nuts_1", "codice_nuts_2"]].nunique() == 1).all().all()))
check("Regional: no missing identifiers (code, name, NUTS, coordinates)",
      int(reg[["codice_regione", "denominazione_regione", "codice_nuts_1", "codice_nuts_2", "lat", "long"]].isna().sum().sum()) == 0)
check("Coordinates inside Italy (lat 35.4-47.2, long 6.5-18.6)",
      bool(reg["lat"].between(35.4, 47.2).all() and reg["long"].between(6.5, 18.6).all()))

# ---- model tables
for name, mt, key in [("Regional", reg_model, ["codice_regione", "date"]), ("National", nat_model, ["date"])]:
    check(f"{name} model table: unique key", not mt.duplicated(key).any())
    new_cols = ["new_deaths", "new_recovered", "new_tests", "nuovi_positivi", "ingressi_terapia_intensiva"]
    neg = sum(int((fnum(mt, c) < 0).sum()) for c in new_cols if c in mt.columns)
    check(f"{name} model table: no negative daily values", neg == 0, f"{neg} negatives")
    numeric = mt.select_dtypes(include=["number"]).astype("float64")
    check(f"{name} model table: no infinite values", bool(np.isfinite(numeric.to_numpy()[~np.isnan(numeric.to_numpy())]).all()))
    check(f"{name} model table: population denominators explicitly marked unusable pending source verification", bool((~mt["population_denominator_usable"].astype(bool)).all()))
check("Regional model table: row count equals clean regional table", len(reg_model) == len(reg))
check("National model table: row count equals clean national table", len(nat_model) == len(nat))
check("Population-based rates remain disabled until source scope is verified",
      bool(reg_model[["new_cases_per_100k", "active_cases_per_100k", "new_cases_per_100k_20plus", "active_cases_per_100k_20plus"]].isna().all().all()))
check("Every populated one-day-ahead target has a next-day date",
      bool(reg_model.loc[reg_model["target_next_day_new_cases"].notna(), "target_date"].notna().all()))
check("Target dates, when present, are exactly one day ahead",
      bool(((reg_model.loc[reg_model["target_date"].notna(), "target_date"] - reg_model.loc[reg_model["target_date"].notna(), "date"]).dt.days == 1).all()))

results = pd.DataFrame(CHECKS)
display(results)
failed = results[results["result"] == "FAIL"]
if failed.empty:
    print(f"\nALL {len(results)} CHECKS PASSED.")
else:
    print(f"\n{len(failed)} CHECK(S) FAILED - read the detail column before using the data:")
    display(failed)

print("\nInformation (not failures):")
print("  corrections applied     :", len(CORRECTIONS))
print("  regional rows flagged identity mismatch:", int(reg["flag_identity_mismatch"].sum()), "(original values kept)")
print("  national rows flagged identity mismatch:", int(nat["flag_identity_mismatch"].sum()), "(original values kept)")

,check,result,detail
0,Population: male + female = total in every row,PASS,0 mismatching rows
1,"Population: (region, age band) key is unique",PASS,
2,Population: every age-band label could be parsed,PASS,
3,Regional: NUTS values in the file agree with t...,PASS,0 conflicts
4,"Regional: no duplicate (region, date) keys",PASS,
5,Regional: complete region x day grid,PASS,0 missing region-days
6,Regional: row count preserved,PASS,37401 vs 37401
7,National: no duplicate dates,PASS,
8,National: complete daily calendar,PASS,0 missing days
9,National: row count preserved,PASS,1781 vs 1781



ALL 35 CHECKS PASSED.

Information (not failures):
  corrections applied     : 3
  regional rows flagged identity mismatch: 14 (original values kept)
  national rows flagged identity mismatch: 12 (original values kept)


## 10. Save everything

Files are written to the `Processed Data` folder shown at the top (change `PROCESSED_DIR_OVERRIDE` in the first cell if you want another location).

In [14]:
# ---- documentation tables
availability = []
for name, frame in [("Regional", reg), ("National", nat)]:
    for c in NUM_COLS:
        if c in frame.columns:
            ok = frame[c].notna()
            availability.append({"dataset": name, "column": c,
                                 "first_available": frame.loc[ok, "date"].min() if ok.any() else pd.NaT,
                                 "last_available": frame.loc[ok, "date"].max() if ok.any() else pd.NaT,
                                 "available_rows": int(ok.sum()), "missing_rows": int((~ok).sum()),
                                 "used_in_model": c not in UNUSABLE_FOR_MODEL})
column_availability = pd.DataFrame(availability)

dictionary = pd.DataFrame([
    ("flag_value_corrected", "True when a value in the row was corrected (see corrections_log.csv)"),
    ("flag_identity_mismatch", "True when an accounting identity still fails (totale_ospedalizzati, totale_positivi, totale_casi); original values kept"),
    ("flag_negative_daily_value", "True when nuovi_positivi or ingressi_terapia_intensiva was negative in the source (set to missing in the model table)"),
    ("flag_cumulative_decrease", "True when a cumulative column decreased vs the previous reported value (source revision)"),
    ("flag_new_cases_mismatch", "True when nuovi_positivi differs from the change in totale_casi"),
    ("new_cases_vs_cumulative_diff", "(change in totale_casi vs previous day) - nuovi_positivi; empty when not comparable"),
    ("new_deaths / new_recovered / new_tests", "Daily change of deceduti / dimessi_guariti / tamponi; consecutive days only; empty if the cumulative decreased"),
    ("new_cases_per_100k / active_cases_per_100k", "Disabled (missing) until the population file scope and reference year are verified."),
    ("new_cases_per_100k_20plus / active_cases_per_100k_20plus", "Disabled (missing) until the population file scope and reference year are verified."),
    ("target_next_day_new_cases", "Next calendar day new positives; supervised label, never include in model features."),
    ("target_date", "Date corresponding to target_next_day_new_cases; label metadata, never a predictor."),
    ("population_observed_sum", "Sum of available age bands; may be incomplete."),
    ("population_scope_verified / population_denominator_usable", "False until the source is confirmed to represent resident population with a known reference year."),
    ("population_total", "Observed population sum where age bands are complete; not approved as an epidemiological denominator until scope/reference year are verified."),
    ("population_20plus", "Residents aged 20+; defined identically for every region"),
    ("flag_population_excludes_young", "True for regions whose age bands start above age 0 (Lombardia: ages 0-4 missing)"),
    ("has_testing_breakdown / has_casi_testati / has_icu_entries", "True from the first date the source column is reported."),
    ("*_value_present", "True only when this exact row contains a non-missing source value."),
    ("has_note / has_note_test / has_note_casi", "True when the source row had a comment / code in that column"),
], columns=["column", "meaning"])

corrections_log = pd.DataFrame(CORRECTIONS, columns=["dataset", "date", "codice_regione", "column", "old_value", "new_value", "rule", "evidence"])
cleaning_report = pd.DataFrame(CLEANING_LOG)

# ---- write
save_csv(reg, "regional_clean.csv")
save_csv(nat, "national_clean.csv")
save_csv(pop, "population_clean.csv")
save_csv(region_dim, "region_dim.csv")
save_csv(notes, "notes_clean.csv")
save_csv(reg_model, "model_regional_daily.csv")
save_csv(nat_model, "model_national_daily.csv")
save_csv(corrections_log, "corrections_log.csv")
save_csv(cleaning_report, "cleaning_report.csv")
save_csv(column_availability, "column_availability.csv")
save_csv(dictionary, "model_columns_dictionary.csv")
save_csv(reconciliation_mismatches, "regional_national_reconciliation_mismatches.csv")
save_csv(reg_identity_left.assign(dataset="Regional"), "remaining_identity_mismatches_regional.csv")
save_csv(nat_identity_left.assign(dataset="National"), "remaining_identity_mismatches_national.csv")
if not reg_decreases.empty:
    save_csv(reg_decreases, "cumulative_decreases_regional.csv")
if not nat_decreases.empty:
    save_csv(nat_decreases, "cumulative_decreases_national.csv")
save_csv(results, "validation_checks.csv")

print("Saved to:", PROCESSED_DIR, "\n")
display(pd.DataFrame(SAVED, columns=["file", "rows", "columns"]))

print("\nCorrections applied:")
display(corrections_log)
print("\nCleaning steps:")
display(cleaning_report)

Saved to: C:\Users\New Star Computer 25\EpiShield-AI\Src\Processed Data 



,file,rows,columns
0,regional_clean.csv,37401,35
1,national_clean.csv,1781,27
2,population_clean.csv,210,15
3,region_dim.csv,21,17
4,notes_clean.csv,1617,3
5,model_regional_daily.csv,37401,52
6,model_national_daily.csv,1781,44
7,corrections_log.csv,3,8
8,cleaning_report.csv,28,4
9,column_availability.csv,38,7



Corrections applied:


,dataset,date,codice_regione,column,old_value,new_value,rule,evidence
0,Regional,2021-11-07,05,totale_positivi,12616,12622,R2+R3,R2 and R3 both failed; totale_positivi recompu...
1,National,2020-02-25,ITA,totale_ospedalizzati,150,149,R1+R2,R1 and R2 both failed; totale_ospedalizzati re...
2,National,2021-11-07,ITA,totale_positivi,96987,96993,R2+R3,R2 and R3 both failed; totale_positivi recompu...



Cleaning steps:


,dataset,step,detail,affected
0,Population,Types and text,"Counts -> Int64, coordinates -> float, text tr...",210.0
1,Population,Region names,Canonical names from population file; mapping ...,NaN
2,Population,Age-band flag,"Age bands with gaps, overlaps, or incomplete a...",1.0
3,Regional,Dropped constant column,'stato' always equals 'ITA',37401.0
4,Regional,Types and text,date -> calendar day; counts -> Int64 (missing...,37401.0
5,Regional,Canonical region names,Names taken from the region dimension via code...,3562.0
6,Regional,Coordinates,lat/long set to one value per region from the ...,37401.0
7,Regional,codice_nuts_1 restored,Empty cells filled from the region dimension (...,6846.0
8,Regional,codice_nuts_2 restored,Empty cells filled from the region dimension (...,6846.0
9,Regional,Evidence-based corrections,Shared faulty field recomputed only when it ma...,1.0


### How to use the output

| Need | Use |
|---|---|
| Train / evaluate a model | `model_regional_daily.csv` (region × day) or `model_national_daily.csv` |
| Faithful cleaned copy of the source | `regional_clean.csv`, `national_clean.csv`, `population_clean.csv`, `notes_clean.csv` |
| Region attributes + population denominators | `region_dim.csv` |
| What was changed and why | `corrections_log.csv`, `cleaning_report.csv` |
| Rows to treat with care | the `flag_*` columns, `remaining_identity_mismatches_*.csv`, `cumulative_decreases_*.csv` |
| When each column starts to exist | `column_availability.csv` and the `has_*` columns |

**Modelling notes**
- Never fill missing values with 0. Use `has_*` flags or restrict training to the period where the column exists.
- Split train / validation / test **by time**, not randomly. All series are time-ordered per region.
- Population DQA also raised an unresolved source-scope/reference-year question (resident population vs target population). Therefore all per-100k rates are deliberately disabled until the source is verified. Population counts remain in `region_dim.csv` for audit/context only; they are intentionally excluded from model tables until verified.
- Large source revisions (e.g. cumulative tests dropping by tens of millions in 2024) are flagged by `flag_cumulative_decrease`; the derived daily series are empty on those days instead of showing impossible values.
- `variazione_totale_positivi` is left exactly as reported, even on the rare day where `totale_positivi` was corrected (the correction is 6 cases or fewer in the data checked).
- `nuovi_positivi` and the change in `totale_casi` disagree on some days (see `new_cases_vs_cumulative_diff`). `nuovi_positivi` is kept as the reported daily series; decide your threshold for large gaps in the modelling phase.

**Forecast target:** `target_next_day_new_cases` is the next calendar day's `nuovi_positivi`. Exclude `target_next_day_new_cases` and `target_date` from predictors. Also do not use same-day fields that would not be known at the intended prediction cutoff. Decide the operational cutoff (e.g. forecast tomorrow after today's report is published) before selecting predictors.

**Population:** `population_observed_sum` records the sum of available age bands. `population_total` is missing when the age bands do not cover all ages (e.g. the identified Lombardia gap). Population-based rates are disabled until the source scope and reference year are verified.

**Reconciliation:** regional-vs-national differences are exported to `regional_national_reconciliation_mismatches.csv`; they are diagnostic and do not automatically fail cleaning or trigger source-value edits.


In [15]:
print("Processed Data Location:")
print(PROCESSED_DIR)

print("\nCSV Files:")
for file in sorted(PROCESSED_DIR.glob("*.csv")):
    print(file.name)

Processed Data Location:
C:\Users\New Star Computer 25\EpiShield-AI\Src\Processed Data

CSV Files:
cleaning_report.csv
column_availability.csv
corrections_log.csv
cumulative_decreases_national.csv
cumulative_decreases_regional.csv
model_columns_dictionary.csv
model_national_daily.csv
model_regional_daily.csv
national_clean.csv
notes_clean.csv
population_clean.csv
region_dim.csv
regional_clean.csv
regional_national_reconciliation_mismatches.csv
remaining_identity_mismatches_national.csv
remaining_identity_mismatches_regional.csv
validation_checks.csv
